In [1]:
import pandas as pd
import sys
import numpy as np
import pandas as pd
import scanpy as sc
import tangram as tg

### Specify path

In [4]:
data_path = "/data/Ajita/Spatial/Datasets/Spatial_Deconvolution/Breast_Atlas/SC/SC.h5ad"
results_path = "/data/Ajita/Spatial/Datasets/Spatial_Deconvolution/Breast_Atlas/simulator_outputs/ST_8/"

ad_sc = sc.read_h5ad(data_path)
print (ad_sc)

celltype_name = "broad_cell_type"

AnnData object with n_obs × n_vars = 714331 × 33234
    obs: 'mapped_reference_assembly', 'mapped_reference_annotation', 'alignment_software', 'donor_id', 'self_reported_ethnicity_ontology_term_id', 'donor_living_at_sample_collection', 'donor_menopausal_status', 'organism_ontology_term_id', 'sample_uuid', 'sample_preservation_method', 'tissue_ontology_term_id', 'development_stage_ontology_term_id', 'sample_derivation_process', 'sample_source', 'donor_BMI_at_collection', 'suspension_depleted_cell_types', 'suspension_derivation_process', 'suspension_dissociation_reagent', 'suspension_dissociation_time', 'suspension_percent_cell_viability', 'suspension_uuid', 'suspension_type', 'library_uuid', 'assay_ontology_term_id', 'sequencing_platform', 'is_primary_data', 'cell_type_ontology_term_id', 'author_cell_type', 'cell_state', 'disease_ontology_term_id', 'sex_ontology_term_id', 'n_count_rna', 'n_feature_rna', 'percent_mito', 'percent_rb', 'tissue_location', 'bmi_group', 'procedure_group', 'ag

In [5]:

sc.pp.normalize_total(ad_sc)
sc.pp.log1p(ad_sc)

celltype_counts = ad_sc.obs[celltype_name].value_counts()
celltype_drop = celltype_counts.index[celltype_counts < 2]

print(f'Drop celltype {list(celltype_drop)} contain less 2 sample')
ad_sc = ad_sc[~ad_sc.obs[celltype_name].isin(celltype_drop),].copy()

sc.tl.rank_genes_groups(ad_sc, groupby = celltype_name, use_raw = False)

markers_df = pd.DataFrame(ad_sc.uns["rank_genes_groups"]["names"]).iloc[0:200, :]
print(markers_df)

genes_sc = np.unique(markers_df.melt().value.values)
print(genes_sc)



Drop celltype [] contain less 2 sample
               basal           bcells      fibroblasts            lumhr  \
0    ENSG00000128422  ENSG00000051523  ENSG00000011465  ENSG00000111057   
1    ENSG00000186847  ENSG00000105369  ENSG00000139329  ENSG00000160862   
2    ENSG00000198467  ENSG00000084070  ENSG00000189058  ENSG00000167642   
3    ENSG00000186081  ENSG00000051108  ENSG00000142173  ENSG00000170421   
4    ENSG00000149591  ENSG00000092820  ENSG00000182326  ENSG00000189334   
..               ...              ...              ...              ...   
195  ENSG00000164587  ENSG00000128218  ENSG00000079819  ENSG00000138448   
196  ENSG00000155755  ENSG00000152256  ENSG00000138758  ENSG00000099246   
197  ENSG00000166833  ENSG00000139266  ENSG00000163453  ENSG00000155115   
198  ENSG00000166619  ENSG00000128590  ENSG00000111678  ENSG00000169083   
199  ENSG00000120738  ENSG00000172071  ENSG00000145824  ENSG00000099282   

              lumsec        lymphatic          myeloid      

### Reading Spatial data

In [8]:
data_path = "/data/Ajita/Spatial/Datasets/Spatial_Deconvolution/Breast_Atlas/simulator_outputs/ST_8/simulated_gene_expression_average.h5ad"

ad_st = sc.read_h5ad(data_path)
genes_st = ad_st.var_names.values
genes = list(set(genes_sc).intersection(set(genes_st)))
tg.pp_adatas(ad_sc, ad_st, genes = genes)

In [ ]:


ad_map = tg.map_cells_to_space(
                   ad_sc,
                   ad_st,
                   mode = 'clusters',
                   num_epochs = 1000,
                   cluster_label = celltype_name)

tg.project_cell_annotations(ad_map, ad_st, annotation = celltype_name)
celltype_density = ad_st.obsm['tangram_ct_pred']
celltype_density = (celltype_density.T/celltype_density.sum(axis=1)).T


celltype_density.to_csv(results_path + 'output_Tangram.csv')



INFO:root:Allocate tensors for mapping.
INFO:root:Begin training with 1376 genes and rna_count_based density_prior in clusters mode...
INFO:root:Printing scores every 100 epochs.


Score: 0.352, KL reg: 0.768
Score: 0.862, KL reg: 0.002
Score: 0.863, KL reg: 0.002
Score: 0.863, KL reg: 0.002
Score: 0.864, KL reg: 0.002
Score: 0.864, KL reg: 0.002
Score: 0.864, KL reg: 0.002
Score: 0.864, KL reg: 0.002
Score: 0.864, KL reg: 0.002
